<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/big_C_cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
%%capture
# 1. Install all dependencies including pandas
!pip install xlsxwriter scrapling patchright msgspec browserforge nest_asyncio polars -q
!patchright install chromium
!patchright install-deps

In [23]:
%%skip
# @title work
# This code works
import asyncio
import nest_asyncio
import pandas as pd
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages():
    base_url = "https://www.bigc.co.th/en/category/laundry?brand=184%2C249%2C188%2C189%2C185"
    current_page = 2
    all_data = []

    # กำหนดค่าภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้า...")

    while True:
        target_url = f"{base_url}&page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True,
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # ถ้าไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A"
                    })

                # เพิ่มเลขหน้าเพื่อไปหน้าถัดไป
                current_page += 1

                # ใส่หน่วงเวลาเล็กน้อยเพื่อไม่ให้โดนบล็อก
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์
    if all_data:
        df = pd.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 2 ถึง {current_page-1}")
        display(df)
        df.to_csv('bigc_laundry_full.csv', index=False)
        print("บันทึกข้อมูลลงไฟล์ bigc_laundry_full.csv เรียบร้อยแล้ว")
    else:
        print("ไม่พบข้อมูลที่จะบันทึก")

# เริ่มทำงาน
await scrape_bigc_multi_pages()

In [30]:
# 1. ติดตั้งไลบรารีที่จำเป็น (รวมถึง polars)
# !pip install scrapling patchright msgspec browserforge nest_asyncio polars -q
# !patchright install chromium
# !patchright install-deps

import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def get_scrape_bigc_multi_pages_polars(base_url):
    current_page = 1
    all_data = []

    # กำหนดค่าคุกกี้และ headers เพื่อรักษาเซสชันภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...")

    while True:
        # ตรวจสอบโครงสร้าง URL เพื่อต่อพารามิเตอร์ page ให้ถูกต้อง
        separator = "&" if "?" in base_url else "?"
        target_url = f"{base_url}{separator}page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            # ใช้ StealthyFetcher เพื่อข้าม Cloudflare WAF
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True, # จัดการกับระบบตรวจสอบบอทอัตโนมัติ
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # หากไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    # สกัดข้อมูลชื่อสินค้าและราคาจาก DOM
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A"
                    })

                current_page += 1
                # หน่วงเวลาเล็กน้อยเพื่อหลีกเลี่ยงการถูกตรวจจับพฤติกรรม
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์และส่งคืนค่าเป็น Polars DataFrame
    if all_data:
        df = pl.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 1 ถึง {current_page-1}")
        return df
    else:
        print("ไม่พบข้อมูลที่จะประมวลผล")
        return pl.DataFrame()

In [ ]:
# เริ่มทำงานและเก็บผลลัพธ์ลงในตัวแปร
df_big_c_laundry = await get_scrape_bigc_multi_pages_polars(base_url="https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100")

if not df_big_c_laundry.is_empty():
    print(df_big_c_laundry.head(10))

เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...
กำลังดึงข้อมูลหน้า 1: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1


[2026-04-08 05:50:57] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.


In [ ]:
df_big_c_dishwash = await get_scrape_bigc_multi_pages_polars(base_url="https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100")
# แสดงตัวอย่างข้อมูล
if not df_big_c_dishwash.is_empty():
    print(df_big_c_dishwash.head(10))

In [27]:
# df_big_c_.write_excel("dishwash_big_c.xlsb")